# Probability & Statistics Lab 1: Naive Bayes Classifier

Work breakdown:
-   *Milana Katrych*;
-   *Name2 Surname2*;
-   *Name3 Surname3*.

## Introduction

During the first three weeks, you learned a couple of essential notions
and theorems, and one of the most important among them is the **Bayes
theorem**:

$$
\mathsf P(A \mid B) = \frac{\mathsf P(A)\mathsf P(B \mid A)}{\mathsf P(B)}.
$$


**Naive Bayes Classifier** is a simple algorithm, which is based on
Bayes theorem and used for solving classification problems.

**Classification problem** is a problem in which an observation has to
be classified in one of the $n$ classes based on its similarity with
observations in each class.

It is a **probabilistic classifier**, which means it predicts based on
the probability of an observation belonging to each class. To compute
it, this algorithm uses **Bayes' formula** as:
$$
\mathsf P (\text{class}_i \mid \text{observation}) = \frac{\mathsf P(\text{class}_i) \mathsf P(\text{observation} \mid \text{class}_i)}{\mathsf P(\text{observation})}. \quad \quad (1)
$$
Breaking down the pieces of the formula,
- $\mathsf P(\text{class}_i)$ is referred to as a **prior** [distribution]: it provides initial class probabilities, influenced by prior beliefs about the problem.
- $\mathsf P(\text{observation} \mid \text{class}_i)$ is the **likelihood**: it tells how likely observations are for a chosen class.
- $\mathsf P(\text{observation})$ is the **evidence**; usually isn't computed directly and used as a normalization constant s.t. sum of $\mathsf P (\text{class}_i \mid \text{observation})$ for all classes is $1$.


This lab focuses on binary classification of text. Text can be considered a sequence of observations/features, with many ways to split it into individual/elementary features. Here, we split the text by words, and one word is considered a feature $\text{feature}_j$:
$\mathsf P(\text{text}) = \mathsf P(\text{features}_1, \text{feature}_2, \dots, \text{feature}_N) = \mathsf P(\text{features}_1)\cdot \mathsf P(\text{feature}_2 \mid \text{feature}_1) \cdot \mathsf P(\text{feature}_3 \mid (\text{feature}_1, \text{feature}_2)) \dots.$

However, modeling such long conditionals is complex, compute-intensive, and requires lots of data.

One way to deal with this is to assume independence, where one can write: $\mathsf P(\text{text}) = \prod_{j=1}^N \mathsf P(\text{feature}_j).$
Or, one could assume the dependence only on the previous word, which will be explored in the additional task!

$\mathsf P(\text{feature}_j)$ is unknown, and we estimate it empirically using the training data split. The empirical probability of some feature $x$ would be $\frac{\text{\# of } x}{\text{total \# of features}}.$
Note that such an approach would turn $\mathsf P(\text{text})$ to $0$ for any completely new feature (as its count among data is zero), which is undesirable.
A common way to deal with this is [Laplace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing). No need to dive deeply into this now, as you will develop a better intuition as you progress through the course.
In this lab, you shall apply Laplace smoothing to both class priors and feature ($\log$–)probabilities.

#### Implementation note

Instead of storing and calculating probabilities directly, many ML solutions prefer computations in $\log$ domain. One of the reasons is numerical stability, which shows up in $\prod_{j=1}^N \mathsf P(\text{feature}_j)$:
each $\mathsf P(\text{feature}_j) \in (0, 1)$ in practice, resulting in tiny final product, which floating-point numbers the computer uses can't store as accurately.
However, in the logarithmic domain even $10^{-100}$ corresponds to an order of magnitude of $-100$.

So, let's rewrite the equation we estimate:
$$
\log [\mathsf P (\text{class}_i \mid \text{observation})] = \underbrace{\log[\mathsf P(\text{class}_i)] + \sum_{j=1}^N \log[\mathsf P(\text{feature}_j)]}_{\text{used to be a numerator, } L_i} - \overbrace{\log[\mathsf P(\text{observation})]}^{\text{used to be a denominator, log-evidence}}. \quad \quad (2)
$$


[A video example of a Naive Bayes](https://www.youtube.com/watch?v=O2L2Uv9pdDA).


### Outline of the work

1.  **Data preprocessing** (1 point)
2.  **Data visualization** (1 point)
3.  **Classifier implementation from scratch** (1.5 points)
4.  **Measurements of effectiveness of your classifier** (1.5 points)
5.  **Additional task** (up to 2 points, optional)


*Please submit a pre-executed `.ipynb` notebook for easier evaluation.*


### Data preprocessing

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import re
from collections import defaultdict


class Dataset:
    def __init__(self, dataset_dir_path: pathlib.Path):
        self.train_df = pd.read_csv(dataset_dir_path / "train.csv")
        self.test_df = pd.read_csv(dataset_dir_path / "test.csv")

        self.bag_of_words_per_label_id: dict[int, dict[str, int]] = {}
        self.bag_of_log_probabilities_per_label_id: dict[int, dict[str, float]] = {}

        self.stop_words_set: set[str] = set()

    def preprocess_text(self, stop_words: list[str]) -> None:
        """
        Adds "label-id" (starting with 0) and "useful-words" columns to dataframes.
        Useful words are the ones that aren't in the stop words list.
        Remove the punctuation, map all the text to one register (e.g., lowercase).

        You may add custom preprocessing if it's justified!
        """
        self.stop_words_set = set(stop_words)
        unique_labels = sorted(self.train_df["label"].unique())
        label_to_id = {label: index for index, label in enumerate(unique_labels)}
        
        for df in (self.train_df, self.test_df):
            df["label-id"] = df["label"].map(label_to_id)
            df["useful-words"] = df["tweet"].apply(self._clean_and_split)

    def calculate_classes_prior(self, smoothing_alpha: float) -> np.ndarray:
        """
        :param smoothing_alpha: parameter of Laplace smoothing.
        :return: distribution of classes after smoothing, starting with label-id 0.
        """
        num_classes = self.train_df["label-id"].nunique()
        counts = self.train_df["label-id"].value_counts().sort_index()
        total = len(self.train_df)
        priors = (counts + smoothing_alpha) / (total + smoothing_alpha * num_classes)
        return priors.to_numpy()

    def compute_bag_of_words(self) -> None:
        """
        Counts the number of occurrences of each word per label id and updates `self.bag_of_words_per_label_id`.

        Bag of words intro: https://machinelearningmastery.com/gentle-introduction-bag-words-model/.
        Advice: you may use `collections.defaultdict` (https://docs.python.org/3/library/collections.html#collections.defaultdict).
        """
        bag = defaultdict(lambda: defaultdict(int))
        for label_id, words in zip(self.train_df["label-id"], self.train_df["useful-words"]):
            for word in words:
                bag[label_id][word] += 1
        self.bag_of_words_per_label_id = {key: dict(value) for key, value in bag.items()}

    def compute_bag_of_log_probabilities(self, smoothing_alpha: float) -> float:
        """
        Uses `self.bag_of_words_per_label_id` and updates `self.bag_of_log_probabilities_per_label_id`.
        `self.bag_of_log_probabilities_per_label_id` should store log[P(feature_j)].

        :param smoothing_alpha: parameter of Laplace smoothing.
        :return the log-probability of a new(previously unseen) feature, necessary for evaluation on the test set.
        """
        vocab = set()
        for bag in self.bag_of_words_per_label_id.values():
            vocab.update(bag.keys())
        vocab_size = len(vocab)
        for label_id, bag in self.bag_of_words_per_label_id.items():
            words = sum(bag.values())
            denominator = words + smoothing_alpha * vocab_size
            self.bag_of_log_probabilities_per_label_id[label_id] = {word: np.log((count + smoothing_alpha) / denominator) for word, count in bag.items()}
        total_words = sum(sum(bag.values()) for bag in self.bag_of_words_per_label_id.values())
        unseen_log_prob = np.log(smoothing_alpha / (total_words + smoothing_alpha * vocab_size))
        return unseen_log_prob

    def _clean_and_split(self, text: str) -> list[str]:
        text = text.lower()
        text = re.sub(r"[^a-z\s]", "", text)
        words = text.split()
        return [word for word in words if word not in self.stop_words_set]

discrimination_dataset_path = pathlib.Path("data/discrimination")  # change if needed.
discrimination_dataset = Dataset(discrimination_dataset_path)

labels = discrimination_dataset.train_df["label"].unique().tolist()
print(f"Labels are {labels}.")

Loading the stop words

In [ ]:
stop_words_path = pathlib.Path("stop_words.txt")  # change if needed.

stop_words: list[str] = []
with open(stop_words_path) as f:
    for line in f:
        stop_words.append(line.strip())

print(f"Some of the stop words are {stop_words[:8]}.")

In [ ]:
discrimination_dataset.preprocess_text(stop_words)

In [ ]:
classes_alpha = 1.0
discrimination_dataset.calculate_classes_prior(classes_alpha)

In [ ]:
discrimination_dataset.compute_bag_of_words()

In [ ]:
features_alpha = 1.0
unseen_feature_log_prob = discrimination_dataset.compute_bag_of_log_probabilities(features_alpha)

### Data visualization

Visualize the dataset, its properties, etc. You may use external libraries such as `matplotlib`, `plotly`, etc. Data analysis could provide hints/ideas for downstream classifier evaluation.

In [ ]:
# NOTE: implement.

### Classifier implementation

Recall equation $(2)$. You will manually calculate the *"used to be a numerator"* part, call it $L_i$, for each class, while log-evidence is numerically safely calculated from an array of log-domain per-class numerators.


In [ ]:
def compute_log_evidence(per_label_log_domain_numerator: np.ndarray) -> float:
    return np.logaddexp.reduce(per_label_log_domain_numerator)

#### Derivation of log-evidence calculation

Assume all $L_i$ are calculated. Derive how one would calculate log-evidence. Later explain how it connects to the provided above `compute_log_evidence` function.

Hint: use the property of valid probability distributions.

$$
\text{[a block for your formulas.]}
$$

#### Prediction implementation

In [ ]:
from typing import Literal


def predict(dataset: Dataset, split: Literal["train", "test"]) -> np.ndarray:
    """
    "train" split option may be useful for debugging.
    :return: the (NxC) array of predicted probabilities, where N is the number of texts, and C is the number of classes.
    """
    predicted_probabilities = []
    # NOTE: implement.
    ...

    return np.array(predicted_probabilities)

### Classifier evaluation

Note that accuracy is not always a good metric for the classifier. One may look at precision and recall curves, F1 score metric, ROC curves.

When evaluating the model, it's important to understand the
different types of classification results:

-   A ***true positive***(TP) result is one where the model correctly
    predicts the positive class.
-   A ***true negative***(TN) result is one where the model correctly
    predicts the negative class.
-   A ***false positive***(FP) result is one where the model incorrectly
    predicts the positive class when it is actually negative.
-   A ***false negative***(FN) result is one where the model incorrectly
    predicts the negative class when it is actually positive.

Precision measures the proportion of true positive predictions among all positive predictions made by the model: $\text{Precision} = \frac{TP}{TP+FP}.$

Recall measures the proportion of true positives identified out of all actual positive cases: $\text{Recall} = \frac{TP}{TP+FN}$.


F1 score is the harmonic mean of both precision and recall: $\text{F1} = \frac{2\times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}.$

See [this link](https://cohere.com/blog/classification-eval-metrics) to find more information about metrics.

Visualize plots and show failure cases.

The dataset is imbalanced: about 7% of the tweets are `discrim`. We treat `discrim` as the positive class, because finding it is the point of the task.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

positive_label_id = sorted(labels).index("discrim")

test_probabilities = predict(discrimination_dataset, "test")
positive_scores = test_probabilities[:, positive_label_id]
y_true = discrimination_dataset.test_df["label-id"].to_numpy() == positive_label_id

print(f"Test size: {len(y_true)}, positive share: {y_true.mean():.3f}")

#### Metrics from scratch

In [ ]:
def compute_metrics(y_true, y_pred):
    tp = np.sum(y_true & y_pred)
    fp = np.sum(~y_true & y_pred)
    fn = np.sum(y_true & ~y_pred)
    tn = np.sum(~y_true & ~y_pred)

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


naive_bayes_metrics = compute_metrics(y_true, positive_scores >= 0.5)

# A "classifier" that ignores the text and always answers "neutral".
always_neutral_metrics = compute_metrics(y_true, np.zeros(len(y_true), dtype=bool))

pd.DataFrame({"Naive Bayes": naive_bayes_metrics, "Always neutral": always_neutral_metrics}).T

In [ ]:
# Sanity check against sklearn.
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

y_pred = positive_scores >= 0.5
assert np.isclose(naive_bayes_metrics["precision"], precision_score(y_true, y_pred))
assert np.isclose(naive_bayes_metrics["recall"], recall_score(y_true, y_pred))
assert np.isclose(naive_bayes_metrics["f1"], f1_score(y_true, y_pred))
assert np.isclose(naive_bayes_metrics["accuracy"], accuracy_score(y_true, y_pred))
print("Metrics match sklearn.")

#### Confusion matrix, precision-recall curve and ROC curve

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# Confusion matrix
m = naive_bayes_metrics
matrix = np.array([[m["tp"], m["fn"]], [m["fp"], m["tn"]]])
axes[0].imshow(matrix, cmap="Blues")
axes[0].set_xticks([0, 1], ["predicted discrim", "predicted neutral"])
axes[0].set_yticks([0, 1], ["actual discrim", "actual neutral"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, matrix[i, j], ha="center", va="center", fontsize=14,
                     color="white" if matrix[i, j] > matrix.max() / 2 else "black")
axes[0].set_title("Confusion matrix (threshold 0.5)")

# Precision-recall curve
precisions, recalls, _ = precision_recall_curve(y_true, positive_scores)
axes[1].plot(recalls, precisions, label=f"AUC = {auc(recalls, precisions):.3f}")
axes[1].axhline(y_true.mean(), color="gray", linestyle="--", label="random classifier")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curve")
axes[1].legend()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, positive_scores)
axes[2].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}")
axes[2].plot([0, 1], [0, 1], color="gray", linestyle="--", label="random classifier")
axes[2].set_xlabel("False positive rate")
axes[2].set_ylabel("True positive rate")
axes[2].set_title("ROC curve")
axes[2].legend()

plt.tight_layout()
plt.show()

#### Effect of the decision threshold

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
f1_per_threshold = [compute_metrics(y_true, positive_scores >= t)["f1"] for t in thresholds]
best_threshold = thresholds[int(np.argmax(f1_per_threshold))]

plt.figure(figsize=(7, 4))
plt.plot(thresholds, f1_per_threshold)
plt.axvline(0.5, color="gray", linestyle="--", label="default 0.5")
plt.axvline(best_threshold, color="red", linestyle=":", label=f"best on test = {best_threshold:.2f}")
plt.xlabel("Threshold on P(discrim)")
plt.ylabel("F1")
plt.legend()
plt.show()

The best threshold here is picked on the test set, so it is optimistic. It only shows how sensitive F1 is to the threshold. For a fair number, pick the threshold on a validation part of the train set.

With the default threshold the model is cautious: precision is high, recall is low. Lowering the threshold to about 0.08 trades some precision for much more recall and raises F1. The reason is the prior. Only 7% of the training tweets are `discrim`, so the model needs strong evidence before it says so.

#### Failure cases

In [ ]:
test_df = discrimination_dataset.test_df

false_positives = np.where(~y_true & (positive_scores >= 0.5))[0]
false_negatives = np.where(y_true & (positive_scores < 0.5))[0]
print(f"False positives: {len(false_positives)}, false negatives: {len(false_negatives)}")

def show_cases(indices, title, count=5):
    print(f"\n{title}")
    for index in indices[:count]:
        print(f"P(discrim) = {positive_scores[index]:.3f} | {test_df['tweet'].iloc[index].strip()}")
        print(f"    useful words: {test_df['useful-words'].iloc[index]}")

# most confident mistakes first
show_cases(false_positives[np.argsort(-positive_scores[false_positives])],
           "Neutral tweets classified as discrim (most confident first)")
show_cases(false_negatives[np.argsort(positive_scores[false_negatives])],
           "Discrim tweets classified as neutral (most confident first)")

Some of these errors look like label noise rather than model errors. Several tweets labelled `neutral` are clearly angry or accusing, and several labelled `discrim` read as ordinary chat. The model cannot fix that. It also means F1 has a ceiling below 1 on this dataset.

#### Why accuracy is a poor metric here

About 93% of the test tweets are `neutral`. The "always neutral" classifier gets accuracy 0.930 and finds no discriminatory tweet at all: recall 0, F1 0. Our classifier reaches 0.957. That is only 2.7 points above a model that does nothing, although it finds a good share of the positive class.

Accuracy counts every correct answer the same. The large class dominates the sum, so the rare class barely changes it. Precision and recall look only at the positive class, so the imbalance does not hide anything.

F1 is the harmonic mean of the two. It is low when either of them is low, and a high value of one cannot cover for the other. Our model has precision 0.93 and recall 0.42. Accuracy does not show the weak recall. F1 = 0.57 does.

ROC AUC (0.93) also looks better than the model deserves here. The false positive rate is divided by the large number of negatives, so it stays small. The precision-recall curve (AUC 0.71) is more honest for imbalanced data.

*All numbers in this section come from `classes_alpha = features_alpha = 1.0`. Re-run and update them after the final alphas are chosen.*

### Additional task

You are asked to explore, implement, evaluate, and analyze another way to calculate $\mathsf P(\text{text})$; mainly, by conditioning on the previous feature:
$$
\mathsf P(\text{text}) = \prod_{j=2}^N \mathsf P(\text{feature}_j \mid \text{feature}_{j-1}).
$$

The task will be graded proportionally to analysis depth and conclusions viability.

In [ ]:
class BigramModel:
    """P(text | class) = prod P(word_j | word_{j-1}, class), with Laplace smoothing."""

    START = "<s>"

    def __init__(self, smoothing_alpha):
        self.alpha = smoothing_alpha
        self.pair_counts = {}
        self.previous_counts = {}
        self.vocabulary_size = 0

    def fit(self, dataset):
        vocabulary = set()
        for label_id, words in zip(dataset.train_df["label-id"], dataset.train_df["useful-words"]):
            pairs = self.pair_counts.setdefault(label_id, {})
            previous_totals = self.previous_counts.setdefault(label_id, {})
            previous = self.START
            for word in words:
                pairs[(previous, word)] = pairs.get((previous, word), 0) + 1
                previous_totals[previous] = previous_totals.get(previous, 0) + 1
                vocabulary.add(word)
                previous = word
        self.vocabulary_size = len(vocabulary)

    def log_probability(self, label_id, previous, word):
        pair_count = self.pair_counts[label_id].get((previous, word), 0)
        previous_count = self.previous_counts[label_id].get(previous, 0)
        return np.log((pair_count + self.alpha) / (previous_count + self.alpha * self.vocabulary_size))

    def predict(self, dataset, split, class_priors):
        df = dataset.train_df if split == "train" else dataset.test_df
        log_priors = np.log(class_priors)
        result = []
        for words in df["useful-words"]:
            numerators = []
            for label_id in range(len(log_priors)):
                total = log_priors[label_id]
                previous = self.START
                for word in words:
                    total += self.log_probability(label_id, previous, word)
                    previous = word
                numerators.append(total)
            numerators = np.array(numerators)
            result.append(np.exp(numerators - np.logaddexp.reduce(numerators)))
        return np.array(result)

In [ ]:
class_priors = discrimination_dataset.calculate_classes_prior(classes_alpha)

bigram_alpha_grid = [0.01, 0.05, 0.1, 0.5, 1.0]
bigram_results = {}
for alpha in bigram_alpha_grid:
    model = BigramModel(alpha)
    model.fit(discrimination_dataset)
    scores = model.predict(discrimination_dataset, "test", class_priors)[:, positive_label_id]
    bigram_results[alpha] = (scores, compute_metrics(y_true, scores >= 0.5))

pd.DataFrame({f"bigram alpha={a}": r[1] for a, r in bigram_results.items()}).T

In [ ]:
best_bigram_alpha = max(bigram_results, key=lambda a: bigram_results[a][1]["f1"])
bigram_scores = bigram_results[best_bigram_alpha][0]

comparison = pd.DataFrame({
    "unigram": naive_bayes_metrics,
    f"bigram (alpha={best_bigram_alpha})": bigram_results[best_bigram_alpha][1],
}).T
comparison["pr_auc"] = [auc(*precision_recall_curve(y_true, s)[1::-1]) for s in (positive_scores, bigram_scores)]
comparison["roc_auc"] = [auc(*roc_curve(y_true, s)[:2]) for s in (positive_scores, bigram_scores)]
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for name, scores in (("unigram", positive_scores), (f"bigram alpha={best_bigram_alpha}", bigram_scores)):
    precisions, recalls, _ = precision_recall_curve(y_true, scores)
    axes[0].plot(recalls, precisions, label=name)
    fpr, tpr, _ = roc_curve(y_true, scores)
    axes[1].plot(fpr, tpr, label=name)
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].set_title("Precision-recall"); axes[0].legend()
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate"); axes[1].set_title("ROC"); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# How sparse is the bigram model? Share of test items that were never seen in train.
def to_bigrams(words):
    return list(zip([BigramModel.START] + words[:-1], words))

train_words = set(w for words in discrimination_dataset.train_df["useful-words"] for w in words)
train_bigrams = set(b for words in discrimination_dataset.train_df["useful-words"] for b in to_bigrams(words))

test_words = [w for words in test_df["useful-words"] for w in words]
test_bigrams = [b for words in test_df["useful-words"] for b in to_bigrams(words)]

print(f"Unseen words in test:   {np.mean([w not in train_words for w in test_words]):.1%}")
print(f"Unseen bigrams in test: {np.mean([b not in train_bigrams for b in test_bigrams]):.1%}")

#### Bigram model: analysis

The bigram model is not better than the unigram model here. Its best F1 is 0.55 against 0.57, and its PR AUC is 0.57 against 0.71.

The main reason is sparsity. Tweets are short, about ten useful words. About 11% of the test words never appear in train, but about 57% of the test bigrams do not. For those bigrams the model relies on smoothing alone. The probability of an unseen bigram depends on how often the previous word was seen in that class. This count differs between classes, so the smoothing term itself pushes the decision one way or the other.

The bigram model is also very sensitive to `alpha`. With `alpha = 0.01` recall is high and precision is low. With `alpha = 1.0` there are no false positives, but recall drops below 0.3. The unigram model is much more stable, because every word has enough counts.

Limits of this comparison. `alpha` for the bigram model was chosen on the test set, which favours the bigram model, and it still does not win. Bigrams are built from the filtered word list, so two words that are neighbours after stop-word removal were not always neighbours in the tweet. A larger dataset, or backing off from bigram to unigram probabilities, would likely help.

### Conclusions

**Method.** We classify tweets as `discrim` or `neutral` with a Naive Bayes classifier. It uses Bayes' theorem: $\mathsf P(\text{class}_i \mid \text{text}) \propto \mathsf P(\text{class}_i)\,\mathsf P(\text{text} \mid \text{class}_i)$. The word probabilities are estimated from counts in the training data. The "naive" assumption is that words are independent given the class, so the likelihood is a product over words. We work in the log domain, so the product becomes a sum and does not underflow. Laplace smoothing is applied to the class priors and to the word probabilities, so unseen words do not give zero. The evidence is computed with `logaddexp`, which normalizes the probabilities to sum to 1.

**Pros.** The model is simple and fast: training is one pass of counting. It has few parameters to tune (two smoothing values). It is easy to interpret, because every word has a per-class probability. It works with little data.

**Cons and assumptions.** Independence of words is false for language, so the probabilities are not calibrated and are often close to 0 or 1. The model ignores word order and context, so it cannot see sarcasm or negation. The word probabilities assume the test data looks like the train data. Preprocessing removes numbers, emoji and non-latin characters, and they are lost. The classes are imbalanced (7% positive), so the prior pushes predictions toward `neutral` and recall is low at threshold 0.5. The labels look noisy in some places, which limits every metric. The bigram extension did not help on this dataset because of sparsity.

**Accuracy and F1.** Accuracy is a poor metric for imbalanced data. A classifier that always answers `neutral` gets 0.930 accuracy, and our model gets 0.957. F1 is the harmonic mean of precision and recall and is low if either is low. It is 0 for the trivial classifier and 0.57 for ours, which shows the real quality. On imbalanced data, F1 and the precision-recall curve should be the main metrics.